In [3]:
pip install pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 3.5 MB/s  0:00:06m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 7.4 MB/s  0:00:01 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

path = "/Users/longquanwen0813/Downloads/images_with_links.csv"

df = pd.read_csv(path, sep=None, engine="python")  # 自动检测分隔符
df.head()

,title,image_url,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,tag_9
0,B751025206PH04PHOT68_bureauJMT_01_7_5_001,https://bibliotheque-numerique.inha.fr/i/?IIIF...,1.4. Matériel de conditionnement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B751025206PH04PHOT68_bureauJMT_01_7_5_002,https://bibliotheque-numerique.inha.fr/i/?IIIF...,1.4. Matériel de conditionnement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B751025206PH04PHOT68_bureauJMT_01_7_5_003,https://bibliotheque-numerique.inha.fr/i/?IIIF...,1.1. Photographie,2.2. Objet,2.2.5 Objet archéologique,2.1.2.4 Statue,NaN,NaN,NaN,NaN,NaN
3,B751025206PH04PHOT68_bureauJMT_01_7_5_004,https://bibliotheque-numerique.inha.fr/i/?IIIF...,2.2. Objet,"1.3 Autre document reproduit (gravure, dessin,...",2.2.5 Objet archéologique,NaN,NaN,NaN,NaN,NaN,NaN
4,B751025206PH04PHOT68_bureauJMT_01_7_5_005,https://bibliotheque-numerique.inha.fr/i/?IIIF...,1.1. Photographie,2.2. Objet,2.2.5 Objet archéologique,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import pandas as pd

tag_cols = [c for c in df.columns if c.startswith("tag")]
tag_cols

['tag_1',
 'tag_2',
 'tag_3',
 'tag_4',
 'tag_5',
 'tag_6',
 'tag_7',
 'tag_8',
 'tag_9']

In [5]:
df["tags_list"] = (
    df[tag_cols]
    .values
    .tolist()
)

# 去掉 None / NaN / 空字符串
df["tags_list"] = df["tags_list"].apply(
    lambda xs: [x for x in xs if pd.notna(x) and str(x).strip() != ""]
)

In [6]:
df["num_labels"] = df["tags_list"].apply(len)

In [7]:
count_by_num = df["num_labels"].value_counts().sort_index()
ratio_by_num = count_by_num / count_by_num.sum()

stats = pd.DataFrame({
    "num_labels": count_by_num.index,
    "num_images": count_by_num.values,
    "proportion": ratio_by_num.values
})

stats

,num_labels,num_images,proportion
0,1,80,0.057845
1,2,79,0.057122
2,3,246,0.177874
3,4,369,0.266811
4,5,401,0.289949
5,6,125,0.090383
6,7,68,0.049168
7,8,13,0.009400
8,9,2,0.001446


In [13]:
q1, q2 = df["num_labels"].quantile([1/3, 2/3])
q1, q2

(4.0, 5.0)

In [10]:
def assign_group_by_quantile(n):
    if n <= q1:
        return "G1_low"
    elif n <= q2:
        return "G2_mid"
    else:
        return "G3_high"

df["label_group"] = df["num_labels"].apply(assign_group_by_quantile)

In [11]:
df.groupby("label_group")["num_labels"].agg(["min", "max", "count"])

,min,max,count
label_group,,,
G1_low,1,4,774
G2_mid,5,5,401
G3_high,6,9,208
